# 2. Encoding Categorical Variables
## Project: Global Life Expectancy Paradox

**Purpose:** Convert the categorical `region` field into numeric indicator columns that machine-learning models can use.

**Method:** One-hot encoding creates one binary column per region. A country receives `1` for its own region and `0` for other regions. The original text label is retained for readability. `name` and `iso2` are identifiers, not ordered categories, so they are not converted into arbitrary numeric ranks.

**Input:** the cleaned country-level CSV.

In [ ]:
# In Google Colab, upload CLEANED_DATASET_FINAL(1).csv when prompted.
from google.colab import files
uploaded = files.upload()
import io, pandas as pd, numpy as np
filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print("Dataset shape:", df.shape)
display(df.head())

## Step 1 — Inspect categories
Check the number of unique regions and their counts before encoding. This is useful for understanding the geographic composition of the dataset.

In [ ]:
print("Unique regions:", df['region'].nunique())
display(df['region'].value_counts().rename_axis('region').reset_index(name='country_count'))

## Step 2 — One-hot encode
The resulting columns use the prefix `region_`. This avoids implying that one region is greater than another, which would happen if region labels were encoded as 0, 1, 2, and so on.

In [ ]:
region_encoded = pd.get_dummies(df['region'], prefix='region', dtype=int)
encoded_df = pd.concat([df.copy(), region_encoded], axis=1)
print("New encoded columns:", len(region_encoded.columns))
display(encoded_df[['name', 'region'] + list(region_encoded.columns[:5])].head())
# Validation: each row should have exactly one active region indicator.
print("Exactly one region active per row:",
      bool((region_encoded.sum(axis=1) == 1).all()))

## Step 3 — Export encoded dataset
The original `region` column remains to keep the dataset understandable. For regression models, you may use `drop_first=True` to omit one reference region when appropriate; retaining all indicators is useful for descriptive analysis and tree-based models.

In [ ]:
output_name = 'life_expectancy_region_encoded.csv'
encoded_df.to_csv(output_name, index=False)
from google.colab import files
files.download(output_name)